# 05 — Classificação binária do IMDb com BERT

**Aluno: Tiago Galhardo Avelar**

Esta é uma cópia resolvida do notebook-modelo da aula. O objetivo é treinar e medir a acurácia de um modelo da família BERT no IMDb, usando 20.000 avaliações para treino e 5.000 para validação.

Requisitos implementados:

- laço próprio de treinamento e validação, sem `Trainer`;
- acumulação explícita de gradientes;
- teste de overfitting em um minibatch;
- melhor checkpoint escolhido pela acurácia de validação;
- avaliação final nas 5.000 amostras reservadas para validação.


## 1. Dependências, configuração e reprodutibilidade

Usamos `prajjwal1/bert-tiny`, uma variante BERT de duas camadas adequada para execução local. A arquitetura e o tokenizador continuam seguindo o padrão BERT uncased. Em GPU compatível, o notebook habilita `bfloat16`; em CPU ou GPU incompatível, usa `float32` automaticamente.


In [1]:
import copy
import math
import os
import random
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch
import transformers
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

SEED = 123
MODEL_NAME = "prajjwal1/bert-tiny"
MAX_LENGTH = 128
MICRO_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH_SIZE = MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS
EPOCHS = 3
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0


def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seeds()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BFLOAT16 = bool(
    DEVICE.type == "cuda"
    and hasattr(torch.cuda, "is_bf16_supported")
    and torch.cuda.is_bf16_supported()
)

print("Dispositivo:", DEVICE)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bfloat16 habilitado:", USE_BFLOAT16)
print("Microbatch:", MICRO_BATCH_SIZE)
print("Passos de acumulação:", GRAD_ACCUM_STEPS)
print("Batch efetivo:", EFFECTIVE_BATCH_SIZE)


C:\workspace\fatec\aprendizagem_maquina\.notebook-deps\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cpu
PyTorch: 2.14.0+cpu
Transformers: 4.57.3
bfloat16 habilitado: False
Microbatch: 32
Passos de acumulação: 2
Batch efetivo: 64


## 2. Dataset IMDb: 20k treino / 5k validação

Os arquivos são ordenados antes da leitura. A divisão do conjunto oficial de treino é embaralhada de forma determinística e contém exatamente 20.000 exemplos de treino e 5.000 de validação.


In [2]:
DATA_DIR = Path("aclImdb")
ARCHIVE = Path("aclImdb_v1.tar.gz")
DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

if not DATA_DIR.exists():
    if not ARCHIVE.exists():
        print("Baixando IMDb...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    print("Extraindo IMDb...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(".")


def load_texts(folder):
    return [
        path.read_text(encoding="utf-8")
        for path in sorted(Path(folder).glob("*.txt"))
    ]


x_train_pos = load_texts(DATA_DIR / "train" / "pos")
x_train_neg = load_texts(DATA_DIR / "train" / "neg")
train_pairs = list(zip(
    x_train_pos + x_train_neg,
    [1] * len(x_train_pos) + [0] * len(x_train_neg),
))
random.Random(SEED).shuffle(train_pairs)

train_pairs = train_pairs[:25_000]
train_pairs, valid_pairs = train_pairs[:20_000], train_pairs[20_000:]
x_train, y_train = map(list, zip(*train_pairs))
x_valid, y_valid = map(list, zip(*valid_pairs))

assert (len(x_train), len(x_valid)) == (20_000, 5_000)
print("Treino:", len(x_train))
print("Validação:", len(x_valid))
print("Distribuição treino:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Distribuição validação:", dict(zip(*np.unique(y_valid, return_counts=True))))


Treino: 20000
Validação: 5000
Distribuição treino: {np.int64(0): np.int64(10027), np.int64(1): np.int64(9973)}
Distribuição validação: {np.int64(0): np.int64(2473), np.int64(1): np.int64(2527)}


## 3. Tokenizador e transformação dos textos

O tokenizador rápido do Hugging Face insere os tokens especiais do BERT, trunca em 128 tokens e aplica padding. A tokenização é feita em blocos para não duplicar todos os textos em memória durante a conversão.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
example = tokenizer(
    "This movie is surprisingly good!",
    truncation=True,
    max_length=MAX_LENGTH,
)
print(example)
print("Decodificado:", tokenizer.decode(example["input_ids"]))


def tokenize_texts(texts, labels, chunk_size=1_000):
    input_ids_parts = []
    attention_mask_parts = []

    for start in range(0, len(texts), chunk_size):
        batch = tokenizer(
            texts[start:start + chunk_size],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        input_ids_parts.append(batch["input_ids"])
        attention_mask_parts.append(batch["attention_mask"])

    return TensorDataset(
        torch.cat(input_ids_parts),
        torch.cat(attention_mask_parts),
        torch.tensor(labels, dtype=torch.long),
    )


print("Tokenizando treino...")
train_dataset = tokenize_texts(x_train, y_train)
print("Tokenizando validação...")
valid_dataset = tokenize_texts(x_valid, y_valid)
sample_ids, sample_mask, sample_label = train_dataset[0]
print("Formas de um exemplo:", sample_ids.shape, sample_mask.shape, sample_label.shape)


{'input_ids': [101, 2023, 3185, 2003, 10889, 2204, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Decodificado: [CLS] this movie is surprisingly good! [SEP]
Tokenizando treino...


Tokenizando validação...


Formas de um exemplo: torch.Size([128]) torch.Size([128]) torch.Size([])


## 4. DataLoaders determinísticos

O `DataLoader` de treino recebe um gerador com seed fixa. O batch efetivo é 64, embora apenas 32 exemplos sejam processados de cada vez: dois microbatches contribuem gradientes antes de cada atualização do otimizador.


In [4]:
def make_loader(dataset, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


train_loader = make_loader(train_dataset, MICRO_BATCH_SIZE, shuffle=True)
valid_loader = make_loader(valid_dataset, MICRO_BATCH_SIZE * 2, shuffle=False)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
print("Microbatches por época:", len(train_loader))
print("Atualizações do otimizador por época:", updates_per_epoch)
print("Batch efetivo:", EFFECTIVE_BATCH_SIZE)


Microbatches por época: 625
Atualizações do otimizador por época: 313
Batch efetivo: 64


## 5. Modelo de classificação

`AutoModelForSequenceClassification` adiciona uma cabeça linear de duas classes sobre o token de classificação do BERT. Todos os parâmetros são ajustados no treino.


In [5]:
def make_model():
    set_seeds()
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )


model = make_model().to(DEVICE)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Parâmetros treináveis: {trainable_parameters:,}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,

## 6. Laço próprio de avaliação

O método abaixo calcula perda e acurácia sem alterar os pesos. A predição é a classe de maior logit.


In [6]:
criterion = nn.CrossEntropyLoss()


def autocast_context():
    return torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.bfloat16,
        enabled=USE_BFLOAT16,
    )


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            attention_mask = attention_mask.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast_context():
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                ).logits
                loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_examples += labels.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


## 7. Laço próprio de treinamento com acumulação de gradientes

O valor da perda é dividido por `gradient_accumulation_steps` antes de `backward()`. Os gradientes são acumulados durante quatro microbatches. `optimizer.step()`, `scheduler.step()` e `zero_grad()` ocorrem apenas ao final de cada grupo — ou no último microbatch da época, caso o total não seja divisível por quatro.


In [7]:
def train_model(
    model,
    train_loader,
    valid_loader,
    epochs,
    learning_rate,
    gradient_accumulation_steps,
    weight_decay=WEIGHT_DECAY,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    total_updates = epochs * math.ceil(
        len(train_loader) / gradient_accumulation_steps
    )
    warmup_steps = int(WARMUP_RATIO * total_updates)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_updates,
    )

    history = []
    best_state = None
    best_valid_accuracy = -1.0
    optimizer_steps = 0
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_examples = 0
        epoch_started = time.perf_counter()

        for micro_step, (input_ids, attention_mask, labels) in enumerate(
            train_loader,
            start=1,
        ):
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            attention_mask = attention_mask.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with autocast_context():
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                ).logits
                full_loss = criterion(logits, labels)
                loss_for_backward = full_loss / gradient_accumulation_steps

            loss_for_backward.backward()
            epoch_loss += full_loss.item() * labels.size(0)
            epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
            epoch_examples += labels.size(0)

            should_update = (
                micro_step % gradient_accumulation_steps == 0
                or micro_step == len(train_loader)
            )
            if should_update:
                nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps += 1

        train_metrics = {
            "loss": epoch_loss / epoch_examples,
            "accuracy": epoch_correct / epoch_examples,
        }
        valid_metrics = evaluate(model, valid_loader)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "valid_loss": valid_metrics["loss"],
            "valid_accuracy": valid_metrics["accuracy"],
            "optimizer_steps": optimizer_steps,
            "seconds": time.perf_counter() - epoch_started,
        }
        history.append(row)
        print(row)

        if valid_metrics["accuracy"] > best_valid_accuracy:
            best_valid_accuracy = valid_metrics["accuracy"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history, optimizer_steps, time.perf_counter() - started


## 8. Verificação de overfitting em um minibatch

Antes do treino completo, uma nova instância do modelo é ajustada repetidamente sobre oito exemplos. Esse teste verifica se modelo, máscara, função de perda, `backward()` e otimizador estão conectados corretamente.


In [8]:
overfit_model = make_model().to(DEVICE)
overfit_optimizer = torch.optim.AdamW(overfit_model.parameters(), lr=5e-4)
fixed_ids, fixed_mask, fixed_labels = next(
    iter(make_loader(train_dataset, batch_size=8, shuffle=False))
)
fixed_ids = fixed_ids.to(DEVICE)
fixed_mask = fixed_mask.to(DEVICE)
fixed_labels = fixed_labels.to(DEVICE)

overfit_history = []
overfit_model.train()
for step in range(1, 61):
    overfit_optimizer.zero_grad(set_to_none=True)
    logits = overfit_model(
        input_ids=fixed_ids,
        attention_mask=fixed_mask,
    ).logits
    loss = criterion(logits, fixed_labels)
    loss.backward()
    overfit_optimizer.step()

    accuracy = (logits.argmax(dim=1) == fixed_labels).float().mean().item()
    overfit_history.append((loss.item(), accuracy))
    if step in {1, 10, 20, 40, 60}:
        print({"step": step, "loss": loss.item(), "accuracy": accuracy})

assert overfit_history[-1][0] < overfit_history[0][0]
assert overfit_history[-1][1] >= 0.875
print("Overfitting do minibatch verificado.")

del overfit_model, overfit_optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'step': 1, 'loss': 0.7644184231758118, 'accuracy': 0.375}


{'step': 10, 'loss': 0.361157089471817, 'accuracy': 1.0}


{'step': 20, 'loss': 0.08003255724906921, 'accuracy': 1.0}


{'step': 40, 'loss': 0.013510055840015411, 'accuracy': 1.0}


{'step': 60, 'loss': 0.006399691104888916, 'accuracy': 1.0}
Overfitting do minibatch verificado.


## 9. Treinamento completo: 20.000 exemplos

O modelo é reinicializado após o teste de overfitting. São executadas três épocas; com quatro microbatches por atualização, a acumulação equivale a batch 128.


In [9]:
model = make_model().to(DEVICE)

history, optimizer_steps, training_seconds = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
)

expected_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
assert optimizer_steps == expected_steps
print("Atualizações esperadas/realizadas:", expected_steps, optimizer_steps)
print(f"Tempo total de treinamento: {training_seconds:.2f} s")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'epoch': 1, 'train_loss': 0.6182366721630096, 'train_accuracy': 0.65445, 'valid_loss': 0.49284915199279783, 'valid_accuracy': 0.7722, 'optimizer_steps': 313, 'seconds': 186.69388099998469}


{'epoch': 2, 'train_loss': 0.4667337596178055, 'train_accuracy': 0.7857, 'valid_loss': 0.4363228302001953, 'valid_accuracy': 0.8032, 'optimizer_steps': 626, 'seconds': 204.46119380003074}


{'epoch': 3, 'train_loss': 0.42926642854213715, 'train_accuracy': 0.80805, 'valid_loss': 0.4288338366508484, 'valid_accuracy': 0.8076, 'optimizer_steps': 939, 'seconds': 275.0024756999919}
Atualizações esperadas/realizadas: 939 939
Tempo total de treinamento: 666.19 s


## 10. Validação final

O melhor estado segundo a acurácia de validação foi restaurado pelo laço de treino. A avaliação abaixo mede a acurácia nas 5.000 amostras reservadas exclusivamente para validação.


In [10]:
best_valid_metrics = evaluate(model, valid_loader)

print("Melhor validação:", best_valid_metrics)

assert best_valid_metrics["accuracy"] >= 0.80, (
    "A acurácia de validação ficou abaixo de 80%; ajuste os hiperparâmetros."
)


Melhor validação: {'loss': 0.4288338366508484, 'accuracy': 0.8076}


## 11. Evidências consolidadas

A célula reúne a configuração da acumulação de gradientes, o teste de overfitting e as métricas finais.


In [11]:
summary = {
    "aluno": "Tiago Galhardo Avelar",
    "modelo": MODEL_NAME,
    "amostras_treino": len(train_dataset),
    "amostras_validacao": len(valid_dataset),
    "micro_batch_size": MICRO_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "batch_efetivo": EFFECTIVE_BATCH_SIZE,
    "optimizer_steps": optimizer_steps,
    "overfit_batch_accuracy": overfit_history[-1][1],
    "melhor_acuracia_validacao": best_valid_metrics["accuracy"],
    "tempo_treino_segundos": training_seconds,
    "bfloat16": USE_BFLOAT16,
}
summary


{'aluno': 'Tiago Galhardo Avelar',
 'modelo': 'prajjwal1/bert-tiny',
 'amostras_treino': 20000,
 'amostras_validacao': 5000,
 'micro_batch_size': 32,
 'gradient_accumulation_steps': 2,
 'batch_efetivo': 64,
 'optimizer_steps': 939,
 'overfit_batch_accuracy': 1.0,
 'melhor_acuracia_validacao': 0.8076,
 'tempo_treino_segundos': 666.1939749000012,
 'bfloat16': False}

## Ferramentas e referências

- **PyTorch** — tensores, diferenciação automática, `DataLoader`, `AdamW` e clipping de gradientes. Documentação: [PyTorch](https://pytorch.org/docs/stable/) e [Automatic Mixed Precision](https://pytorch.org/docs/stable/amp.html).
- **Hugging Face Transformers** — tokenizador, pesos pré-treinados, classificador e scheduler. Documentação: [sequence classification](https://huggingface.co/docs/transformers/tasks/sequence_classification) e [optimization](https://huggingface.co/docs/transformers/main_classes/optimizer_schedules).
- **BERT tiny** — modelo `prajjwal1/bert-tiny`: [model card](https://huggingface.co/prajjwal1/bert-tiny).
- **BERT** — Devlin et al., *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*: [artigo](https://aclanthology.org/N19-1423/).
- **IMDb Large Movie Review Dataset** — Maas et al., *Learning Word Vectors for Sentiment Analysis*: [artigo](https://aclanthology.org/P11-1015/) e [dataset](https://ai.stanford.edu/~amaas/data/sentiment/).
- **Notebook-modelo da aula** — estrutura de dataset, overfitting em um batch e pontos de extensão: [Google Colab](https://colab.research.google.com/drive/1yIJyAQ8nrfgm1zeApdsdFmv6AJWxranF?usp=sharing&authuser=0).
- **Jupyter Notebook / nbclient** — execução das células e preservação das saídas no `.ipynb`: [nbclient](https://nbclient.readthedocs.io/).


## Comentários

---
**Comentários:**

Ferramenta: OpenAI Codex (modelo GPT-5)

Utilização: Utilizado como apoio para interpretar o notebook-modelo, desenvolver e revisar o laço próprio de treinamento, implementar a acumulação de gradientes, estruturar o teste de overfitting, documentar as referências e validar a reprodutibilidade. As métricas apresentadas são produzidas pela execução efetiva das células.

Ferramenta: Hugging Face Transformers

Utilização: Utilizado para carregar o tokenizador e os pesos pré-treinados da variante BERT, além do scheduler de taxa de aprendizado.

Ferramenta: PyTorch

Utilização: Utilizado para o modelo, diferenciação automática, acumulação de gradientes, otimização, DataLoaders e avaliação da acurácia.

Ferramenta: Jupyter Notebook / nbclient

Utilização: Utilizado para executar o notebook localmente e registrar as evidências no arquivo entregue.
---


## Conclusão

O modelo BERT foi ajustado para classificação binária no IMDb com um laço de treinamento próprio. A acumulação de gradientes permitiu obter batch efetivo maior sem manter todos os exemplos simultaneamente na memória. O teste em um minibatch confirmou a capacidade de overfitting, e as métricas finais documentam o desempenho em 20 mil exemplos de treino e 5 mil de validação.
